In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('train.txt', sep =';', header = None, names = ['text', 'emotion'])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

,0
text,0
emotion,0


In [5]:
unqiue_emotions = df['emotion'].unique()
emotion_number = {}
i = 0
for emotion in unqiue_emotions:
    emotion_number[emotion] = i
    i += 1

df['emotion'] =df['emotion'].map(emotion_number)

In [6]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


1. Convert to lower case

In [7]:
df['text'] = df['text'].apply(lambda x: x.lower())

2. Remove punctuations

In [8]:
import string

def remove_punctuation(txt):
    return txt.translate(str.maketrans('', '', string.punctuation))

df['text'] = df['text'].apply(remove_punctuation)

3. Remove numbers

In [9]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

4. Remove emoji & special characters

In [10]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

SPACY  NLTK


STOPWORDS REMOVAL

In [11]:
import nltk

In [12]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [13]:
stop_words = set(stopwords.words('english'))

In [14]:
def remove(txt):
    word = word_tokenize(txt)
    cleaned = []
    for i in word:
        if i not in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)

In [15]:
df['text'] = df['text'].apply(remove)

In [16]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [18]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))


0.7678125


In [19]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

In [21]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [22]:
y_pred = nb2_model.predict(X_test_tfidf)

In [23]:
print(accuracy_score(y_test, y_pred))

0.6609375


In [24]:
from sklearn.linear_model import LogisticRegression

In [25]:
logistic_model = LogisticRegression(max_iter=1000)

In [26]:
logistic_model.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [27]:
log_pred = logistic_model.predict(X_test_tfidf)

In [28]:
print(accuracy_score(y_test,log_pred ))

0.8615625
